# VGGT-1B vs VGGT-Omega on MARS-LVIG HKairport_GNSS03 (RTK ground truth)

A plain model benchmark - no RTVIO pipeline, no window stitching. The same video clips from the drone flight go through each model in **one forward pass**, and the predicted camera path is scored against the drone's RTK track.

Per model, clip and clip length:
- **ATE (Sim3 to RTK)** - RMSE of camera centres after the best similarity fit to RTK: pure trajectory-shape quality, in metres and as % of the clip's path length.
- **RPE** - mean error of each frame-to-frame step after that fit: local consistency.
- **GPS-anchored error** - the fit is made to the **standalone u-blox GPS** instead (what a real flight has), then scored against RTK: the practical number for the 1 m target. The constant u-blox-vs-RTK reference offset is removed first.
- **Speed and memory** - forward-pass seconds, frames/s, peak VRAM.

### Runs locally (e.g. RTX 5070 Ti, 16 GB) or on Kaggle - Windows or Linux

**Local, once:**
1. Python 3.10-3.12 environment with a CUDA build of PyTorch. An RTX 50-series (Blackwell, sm_120) needs **PyTorch >= 2.7 built for CUDA 12.8**:
   `pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128`
2. `pip install rosbags einops safetensors huggingface_hub pandas matplotlib jupyter` and have `git` on PATH.
3. Get `HKairport_GNSS03.bag` (9.09 GB) and set `BAG` in the settings cell. Checkpoints: set their paths, or leave `None` to download from Hugging Face (VGGT-Omega is gated - run `huggingface-cli login` with your approved account, or set the `HF_TOKEN` environment variable).

**Kaggle:** GPU T4 x2, Internet On, attach the bag dataset and the two model inputs; paths are found automatically.

Disk: ~0.6 GB of extracted frames in `WORK`. VRAM: 64-frame clips fit in 16 GB in bf16/fp16; lower `CLIP_LENS` if a model reports OOM.

In [ ]:
import gc, glob, json, math, os, subprocess, sys, time
from pathlib import Path
import numpy as np
import torch

ON_KAGGLE = os.path.isdir("/kaggle/input")
assert torch.cuda.is_available(), "no CUDA GPU visible to PyTorch - install a CUDA build of torch (see above)"
cap = torch.cuda.get_device_capability(0)
print("torch %s (CUDA %s) on %s, sm_%d%d" % (torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), *cap))
if "sm_%d%d" % cap not in torch.cuda.get_arch_list():
    print("WARNING: this torch build has no kernels for sm_%d%d (%s) - on an RTX 50-series install the cu128 "
          "build of PyTorch >= 2.7" % (cap + (torch.cuda.get_arch_list(),)))

missing = [m for m in ("rosbags", "einops", "safetensors", "huggingface_hub", "pandas", "matplotlib")
           if __import__("importlib").util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)

## Settings

- `BAG` / `CKPT_PATHS`: `None` = search `SEARCH_DIRS` (Kaggle: `/kaggle/input`), then download checkpoints from Hugging Face. Use raw strings for Windows paths, e.g. `r"D:\data\HKairport_GNSS03.bag"`.
- `STEP = 2` keeps 5 of the camera's 10 frames/s (the RTVIO benchmark's rate).
- `CLIP_LENS`: frames per forward pass. Each clip is scored at every length (prefixes of the same clip), so the table shows how error grows with sequence length.
- `N_CLIPS` clips are spread evenly over the flight (takeoff/landing trimmed).
- `PRECISION`: bf16 where the GPU supports it (RTX 30/40/50, A100...), else fp16 (T4); `None` = fp32. NaN poses are reported, not dropped.

In [ ]:
BAG = None                                   # e.g. r"D:\data\HKairport_GNSS03.bag"
CKPT_PATHS = {"vggt-1b": None, "vggt-omega": None}
SEARCH_DIRS = ["/kaggle/input"] if ON_KAGGLE else [os.getcwd(), str(Path.home() / "Downloads")]
WORK = "/kaggle/working/model_bench" if ON_KAGGLE else os.path.abspath("model_bench")

STEP = 2
CLIP_LENS = [16, 32, 64]
N_CLIPS = 6
MODELS_TO_RUN = ["vggt-1b", "vggt-omega"]
AUTO = torch.bfloat16 if torch.cuda.is_bf16_supported() and cap[0] >= 8 else torch.float16
PRECISION = {"vggt-1b": AUTO, "vggt-omega": AUTO}
DEVICE = "cuda:0"

OUT = WORK
FRAMES = os.path.join("/tmp" if ON_KAGGLE else WORK, "frames")
REPO_DIR = os.path.join(WORK, "repos")
for d in (OUT, FRAMES, REPO_DIR):
    os.makedirs(d, exist_ok=True)

REPOS = {"vggt": "https://github.com/facebookresearch/vggt.git",
         "vggt-omega": "https://github.com/facebookresearch/vggt-omega.git"}
for name, url in REPOS.items():
    d = os.path.join(REPO_DIR, name)
    if not os.path.isdir(d):
        subprocess.run(["git", "clone", "-q", "--depth", "1", url, d], check=True)
    sys.path.insert(0, d)          # no `pip install -e`: their pinned requirements could replace your torch
    print(name, subprocess.run(["git", "-C", d, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
print("precision:", {k: str(v) for k, v in PRECISION.items()}, "\nwork dir:", WORK)

## 1. Checkpoints and bag

In [ ]:
def find(pattern):
    for root in SEARCH_DIRS:
        hits = glob.glob(os.path.join(root, "**", pattern), recursive=True)
        if hits:
            return hits[0]
    return None

def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"].strip()
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return (UserSecretsClient().get_secret("HF_TOKEN") or "").strip() or None
        except Exception:
            pass
    return None                                  # falls back to a `huggingface-cli login` token, if any

def checkpoint(name, pattern, repo, filename):
    p = CKPT_PATHS.get(name) or find(pattern)
    if p:
        return p
    from huggingface_hub import hf_hub_download
    print("  %s not found locally - downloading %s/%s" % (pattern, repo, filename))
    return hf_hub_download(repo, filename, token=hf_token(), local_dir=os.path.join(WORK, "ckpt"))

CKPT = {}
if "vggt-1b" in MODELS_TO_RUN:
    CKPT["vggt-1b"] = checkpoint("vggt-1b", "vggt1b_model.pt", "facebook/VGGT-1B", "model.pt")
if "vggt-omega" in MODELS_TO_RUN:
    CKPT["vggt-omega"] = checkpoint("vggt-omega", "vggt_omega_1b_512.pt", "facebook/VGGT-Omega", "vggt_omega_1b_512.pt")
for k, v in CKPT.items():
    print("%-11s %s (%.2f GB)" % (k, v, os.path.getsize(v) / 1e9))

BAG = BAG or find("HKairport_GNSS03.bag")
assert BAG and os.path.isfile(BAG), "HKairport_GNSS03.bag not found - set BAG in the settings cell"
print("bag", BAG)

## 2. Ground truth, GPS and clip frames from the bag

Clocks, as verified on this bag by `rtvio/tools/marslvig_to_session.py`: camera and RTK header stamps are UTC; the u-blox time comes from GPS week / time-of-week, which is 18 s ahead of UTC (leap seconds) and is corrected here.

In [ ]:
from rosbags.highlevel import AnyReader

T_CAMERA, T_RTK, T_PVT = "/left_camera/image/compressed", "/dji_osdk_ros/rtk_position", "/ublox_driver/receiver_pvt"
GPS_EPOCH_UNIX, LEAP_S = 315964800, 18

def stamp(m):
    return m.header.stamp.sec + m.header.stamp.nanosec * 1e-9

def ecef(lla):
    a, e2 = 6378137.0, 6.69437999014e-3
    lat, lon, h = np.radians(lla[:, 0]), np.radians(lla[:, 1]), lla[:, 2]
    n = a / np.sqrt(1 - e2 * np.sin(lat) ** 2)
    return np.stack([(n + h) * np.cos(lat) * np.cos(lon), (n + h) * np.cos(lat) * np.sin(lon),
                     (n * (1 - e2) + h) * np.sin(lat)], 1)

def to_enu(lla, ref):
    lat, lon = np.radians(ref[0]), np.radians(ref[1])
    R = np.array([[-np.sin(lon), np.cos(lon), 0],
                  [-np.sin(lat) * np.cos(lon), -np.sin(lat) * np.sin(lon), np.cos(lat)],
                  [np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)]])
    return (ecef(lla) - ecef(np.asarray(ref, float)[None])) @ R.T

t0 = time.time()
with AnyReader([Path(BAG)]) as r:
    conns = {c.topic: c for c in r.connections}
    def read(topic):
        for c, _, raw in r.messages(connections=[conns[topic]]):
            yield r.deserialize(raw, c.msgtype)

    rtk = np.array([[stamp(m), m.latitude, m.longitude, m.altitude] for m in read(T_RTK)])
    gnss = np.array([[GPS_EPOCH_UNIX + m.time.week * 604800 + m.time.tow - LEAP_S, m.latitude, m.longitude, m.altitude]
                     for m in read(T_PVT) if m.valid_fix and m.fix_type >= 3])
    rtk, gnss = rtk[np.argsort(rtk[:, 0])], gnss[np.argsort(gnss[:, 0])]
    REF = tuple(rtk[0, 1:])
    rtk_enu, gnss_enu = to_enu(rtk[:, 1:], REF), to_enu(gnss[:, 1:], REF)

    # Flight = RTK horizontal speed > 1 m/s (drops takeoff, hover and landing), +-2 s.
    v = np.linalg.norm(np.diff(rtk_enu[:, :2], axis=0), axis=1) / np.maximum(np.diff(rtk[:, 0]), 1e-3)
    moving = np.flatnonzero(v > 1.0)
    seg = (rtk[moving[0], 0] - 2.0, rtk[moving[-1] + 1, 0] + 2.0)

    # Clip windows (in time) spread over the flight; each holds max(CLIP_LENS) frames at STEP.
    dur = max(CLIP_LENS) * STEP / 10.0
    starts = np.linspace(seg[0], seg[1] - dur - 0.5, N_CLIPS)
    clip_frames = [[] for _ in starts]                  # [(t, path)] per clip
    seen = [0] * len(starts)
    for m in read(T_CAMERA):
        t = stamp(m)
        for k, s in enumerate(starts):
            if s <= t < s + dur + 0.5 and len(clip_frames[k]) < max(CLIP_LENS):
                if seen[k] % STEP == 0:
                    p = os.path.join(FRAMES, "c%02d_%04d.jpg" % (k, len(clip_frames[k])))
                    with open(p, "wb") as f:
                        f.write(bytes(m.data))
                    clip_frames[k].append((t, p))
                seen[k] += 1

print("read bag in %.0f s: %d RTK, %d u-blox fixes, flight %.0f s" % (time.time() - t0, len(rtk), len(gnss), seg[1] - seg[0]))
for k, cf in enumerate(clip_frames):
    print("clip %d: %d frames, t = %.1f .. %.1f s into the flight" % (k, len(cf), cf[0][0] - seg[0], cf[-1][0] - seg[0]))
assert all(len(cf) == max(CLIP_LENS) for cf in clip_frames), "a clip came out short - lower N_CLIPS or CLIP_LENS"

def interp(t_q, t, xyz, max_gap=0.5):
    j = np.clip(np.searchsorted(t, t_q), 1, len(t) - 1)
    ok = (t_q >= t[0]) & (t_q <= t[-1]) & (t[j] - t[j - 1] <= max_gap)
    w = ((t_q - t[j - 1]) / np.maximum(t[j] - t[j - 1], 1e-9))[:, None]
    return xyz[j - 1] * (1 - w) + xyz[j] * w, ok

# The two receivers disagree by a constant offset (reference frames / altitude datum): measured over the
# whole flight and removed from the u-blox track, so GPS-anchored scores measure the model, not the datum.
in_seg = (gnss[:, 0] >= seg[0]) & (gnss[:, 0] <= seg[1])
rtk_at_gnss, ok = interp(gnss[in_seg, 0], rtk[:, 0], rtk_enu)
OFFSET = np.median(rtk_at_gnss[ok] - gnss_enu[in_seg][ok], axis=0)
gnss_fixed = gnss_enu + OFFSET
print("RTK - u-blox offset removed (E, N, U): %.2f, %.2f, %.2f m" % tuple(OFFSET))

## 3. Models

In [ ]:
def autocast(dtype):
    return torch.autocast("cuda", dtype=dtype or torch.float16, enabled=dtype is not None)

def load_model(name, device):
    if name == "vggt-1b":
        from vggt.models.vggt import VGGT
        m = VGGT()
    else:
        from vggt_omega.models import VGGTOmega
        m = VGGTOmega()
    m.load_state_dict(torch.load(CKPT[name], map_location="cpu", mmap=True, weights_only=True))
    return m.to(device).eval()

def run_model(name, model, paths, device, dtype):
    # Returns (camera centres S x 3 in the model's frame, forward seconds).
    if name == "vggt-1b":
        from vggt.utils.load_fn import load_and_preprocess_images
        from vggt.utils.pose_enc import pose_encoding_to_extri_intri as decode
        imgs = load_and_preprocess_images(paths).to(device)
    else:
        from vggt_omega.utils.load_fn import load_and_preprocess_images
        from vggt_omega.utils.pose_enc import encoding_to_camera as decode
        imgs = load_and_preprocess_images(paths, image_resolution=512).to(device)
    torch.cuda.synchronize(device)
    t = time.time()
    with torch.inference_mode(), autocast(dtype):
        pred = model(imgs)
    torch.cuda.synchronize(device)
    dt = time.time() - t
    extr = decode(pred["pose_enc"].float(), imgs.shape[-2:])[0].reshape(-1, 3, 4).double().cpu()
    R, tr = extr[:, :, :3], extr[:, :, 3]
    C = -torch.einsum("sji,sj->si", R, tr)          # camera-from-world (OpenCV) -> camera centre
    return C.numpy(), dt, tuple(imgs.shape[-2:])

## 4. Scoring

In [ ]:
def umeyama(src, dst):
    mu_s, mu_d = src.mean(0), dst.mean(0)
    xs, xd = src - mu_s, dst - mu_d
    U, D, Vt = np.linalg.svd(xd.T @ xs / len(src))
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[2, 2] = -1
    R = U @ S @ Vt
    s = np.trace(np.diag(D) @ S) / (xs ** 2).sum(1).mean()
    return s, R, mu_d - s * R @ mu_s

def score(C, t_frames):
    gt, ok_gt = interp(t_frames, rtk[:, 0], rtk_enu)
    gps, ok_gps = interp(t_frames, gnss[:, 0], gnss_fixed, max_gap=1.0)
    ok = ok_gt & np.isfinite(C).all(1)
    C, gt, gps, ok_gps = C[ok], gt[ok], gps[ok], ok_gps[ok]
    path = np.linalg.norm(np.diff(gt, axis=0), axis=1).sum()
    s, R, t = umeyama(C, gt)
    a = s * C @ R.T + t
    e = np.linalg.norm(a - gt, axis=1)
    rpe = np.linalg.norm(np.diff(a, axis=0) - np.diff(gt, axis=0), axis=1)
    s2, R2, t2 = umeyama(C[ok_gps], gps[ok_gps])
    g = s2 * C @ R2.T + t2
    eg = g - gt
    return dict(frames=int(ok.sum()), path_m=path, ate_rmse_m=float(np.sqrt((e ** 2).mean())),
                ate_pct=100 * float(np.sqrt((e ** 2).mean())) / max(path, 1e-9), rpe_mean_m=float(rpe.mean()),
                gps_h_median_m=float(np.median(np.linalg.norm(eg[:, :2], axis=1))),
                gps_3d_rmse_m=float(np.sqrt((np.linalg.norm(eg, axis=1) ** 2).mean())),
                gps_within_1m=float((np.linalg.norm(eg, axis=1) <= 1.0).mean())), a, gt

# Reference floor: the u-blox fixes themselves (offset removed) against RTK at the same frame times.
floor = []
for cf in clip_frames:
    tq = np.array([t for t, _ in cf])
    g, ok1 = interp(tq, gnss[:, 0], gnss_fixed, max_gap=1.0)
    gt, ok2 = interp(tq, rtk[:, 0], rtk_enu)
    floor.append(np.linalg.norm((g - gt)[ok1 & ok2], axis=1))
print("u-blox vs RTK at frame times (offset removed): 3D median %.2f m, p90 %.2f m"
      % (np.median(np.concatenate(floor)), np.percentile(np.concatenate(floor), 90)))

## 5. Run

In [ ]:
rows, tracks = [], {}
for name in MODELS_TO_RUN:
    torch.cuda.empty_cache()
    model = load_model(name, DEVICE)
    run_model(name, model, [p for _, p in clip_frames[0][:4]], DEVICE, PRECISION[name])     # warm-up
    for L in CLIP_LENS:
        for k, cf in enumerate(clip_frames):
            tq = np.array([t for t, _ in cf[:L]])
            torch.cuda.reset_peak_memory_stats(DEVICE)
            row = dict(model=name, clip=k, frames_in=L, precision=str(PRECISION[name]).replace("torch.", ""))
            try:
                C, dt, hw = run_model(name, model, [p for _, p in cf[:L]], DEVICE, PRECISION[name])
                row.update(forward_s=dt, fps=L / dt, peak_vram_gb=torch.cuda.max_memory_allocated(DEVICE) / 1e9,
                           input_hw="%dx%d" % hw)
                if not np.isfinite(C).all():
                    row["status"] = "NaN poses"
                else:
                    sc, a, gt = score(C, tq)
                    row.update(sc, status="ok")
                    if L == max(CLIP_LENS):
                        tracks[(name, k)] = (a, gt)
            except torch.cuda.OutOfMemoryError:
                row["status"] = "OOM"
                torch.cuda.empty_cache()
            rows.append(row)
            print({k2: (round(v, 3) if isinstance(v, float) else v) for k2, v in row.items()})
    del model
    gc.collect()
    torch.cuda.empty_cache()

json.dump(rows, open(os.path.join(OUT, "results.json"), "w"), indent=1)

## 6. Results

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT, "results.csv"), index=False)
ok = df[df.status == "ok"]
summary = ok.groupby(["model", "frames_in"]).agg(
    clips=("clip", "count"), ate_rmse_m=("ate_rmse_m", "median"), ate_pct=("ate_pct", "median"),
    rpe_mean_m=("rpe_mean_m", "median"), gps_h_median_m=("gps_h_median_m", "median"),
    gps_within_1m=("gps_within_1m", "mean"), fps=("fps", "median"), peak_vram_gb=("peak_vram_gb", "max")).round(3)
failed = df[df.status != "ok"].groupby(["model", "frames_in", "status"]).size()
summary.to_csv(os.path.join(OUT, "summary.csv"))
print("Medians over clips (gps_within_1m = mean share of frames):\n")
try:
    print(summary.to_markdown())
except ImportError:          # to_markdown needs tabulate
    print(summary.to_string())
if len(failed):
    print("\nFailed runs:\n", failed.to_string())

In [ ]:
import matplotlib.pyplot as plt
colors = {"vggt-1b": "tab:red", "vggt-omega": "tab:blue"}
fig, axes = plt.subplots(2, math.ceil(N_CLIPS / 2), figsize=(5 * math.ceil(N_CLIPS / 2), 10))
for k, ax in enumerate(axes.flat[:N_CLIPS]):
    gt = None
    for name in MODELS_TO_RUN:
        if (name, k) in tracks:
            a, gt = tracks[(name, k)]
            ax.plot(a[:, 0], a[:, 1], "-", color=colors.get(name), lw=1.2, label=name)
    if gt is not None:
        ax.plot(gt[:, 0], gt[:, 1], "k-", lw=2, label="RTK", zorder=0)
        ax.plot(gt[0, 0], gt[0, 1], "go")
    ax.set_title("clip %d (%d frames, Sim3 to RTK)" % (k, max(CLIP_LENS)))
    ax.set_aspect("equal")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "clips_vs_rtk.png"), dpi=100)
plt.show()